# 第 6 章：Transformer Block

这个 notebook 对应 `lessons/06_transformer_block.md`，演示 multi-head causal attention、pre-norm residual、FFN、dropout train/eval 差异，以及堆叠 block 的反传。

In [ ]:
import torch

from src.models.transformer_block import (
    CausalSelfAttention,
    FeedForward,
    TransformerBlock,
)

## 1. Multi-Head Shape

`hidden_dim` 被拆成多个 head，但 attention 输出会投影回原来的 hidden_dim。

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 5, 12)
attention = CausalSelfAttention(hidden_dim=12, num_heads=3)
attn_output = attention(x, return_weights=True)

print("values:", attn_output.values.shape)
print("weights:", attn_output.weights.shape)

## 2. Causal Mask 对所有 Head 生效

未来位置的权重应在 batch 和所有 head 上都为 0。

In [ ]:
future = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
future_weights = attn_output.weights.masked_select(future.view(1, 1, 5, 5))
print("future mass:", future_weights.sum().item())

## 3. FeedForward 与 Transformer Block

FFN 是逐位置 MLP。Pre-norm block 用 `x + module(LayerNorm(x))` 保留 residual 路径。

In [ ]:
ffn = FeedForward(hidden_dim=12)
block = TransformerBlock(hidden_dim=12, num_heads=3)
print("ffn:", ffn(x).shape)
print("block:", block(x).shape)

## 4. Dropout 的 train/eval 差异

dropout 在 train 模式随机丢弃，在 eval 模式保持确定。

In [ ]:
drop_block = TransformerBlock(hidden_dim=12, num_heads=3, dropout=0.5)
drop_block.train()
train_a = drop_block(x)
train_b = drop_block(x)

drop_block.eval()
eval_a = drop_block(x)
eval_b = drop_block(x)

print("train deterministic:", torch.allclose(train_a, train_b))
print("eval deterministic:", torch.allclose(eval_a, eval_b))

## 5. 堆叠 Block 后反传

多个 block 堆叠后，输出不应出现 NaN，输入和参数都应收到梯度。

In [ ]:
stack = torch.nn.Sequential(
    TransformerBlock(hidden_dim=12, num_heads=3),
    TransformerBlock(hidden_dim=12, num_heads=3),
)
x_with_grad = x.detach().clone().requires_grad_(True)
out = stack(x_with_grad)
loss = out.pow(2).mean()
loss.backward()

grad_norm = x_with_grad.grad.norm().item()
print("has nan:", torch.isnan(out).any().item())
print("input grad norm:", round(grad_norm, 6))